### Variables

In [52]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics.pairwise import cosine_distances

RESUME_ID = "72b39379-e2da-4b28-9102-a32b77eacd97"
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.66
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.66
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3

LLM_MODEL_VECTOR_DIMENSIONS = 3072
DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

### Functions Definitions

In [53]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def create_match_score_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

def analyze_market(market_obj: MarketSkillsMatrix, candidate_skills_df: pd.DataFrame, skills_type: str) -> None:
    weight_column_index = SOFT_SKILLS_WEIGHT_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_WEIGHT_COLUMN_INDEX
    string_column_index = SOFT_SKILLS_STRING_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_STRING_COLUMN_INDEX
    threshold = SOFT_SKILLS_SIMILARITY_THRESHOLD if skills_type == "soft" else HARD_SKILLS_SIMILARITY_THRESHOLD
    skills_count = candidate_skills_df.shape[0]

    for i in range(0, skills_count):
        weight = candidate_skills_df.iloc[(i, weight_column_index)] 
        skill_embedding = candidate_skills_df.iloc[(i, string_column_index) ] 
        cosine_similarities = cosine_similarities_matrix(skill_embedding, market_obj.embedding_matrix)
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
        market_obj.accumulate_matches(binary_mask)
        market_obj.accumulate_weighted_matches(weight, binary_mask)

def build_padded_matrix(
        df: pd.DataFrame,
        column_name: str,
        length: int,
        pad_value,
        dtype=None,
    ) -> np.ndarray:
        rows = [
            np.pad(
                array=group[column_name].values,
                pad_width=(0, length - len(group)),
                constant_values=pad_value,
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=dtype)

def build_embedding_matrix(
    df: pd.DataFrame, max_skills: int
    ) -> np.ndarray:
        zero_vector = np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)
        rows = [
            np.vstack(
                list(group["embedding"].values)
                + [zero_vector] * (max_skills - len(group))
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=np.float32)

def get_market_analysis_results(market_obj: MarketSkillsMatrix) -> list[dict]:
    compliance_by_job = market_obj.get_min_compliance_pct_by_job()
    ideal_compliance_by_job = market_obj.get_ideal_compliance_pct_by_job()
    noncompliance_mask = market_obj._match_score_matrix == 1

    return [
        {
            "job_index": i,
            "job_id": job_id,
            "minimum_compliance_pct": compliance_pct,
            "ideal_compliance_pct": ideal_compliance_pct,

            "nonmatched_skills_count": int(noncompliance_mask[i].sum()),
            "nonmatched_skills": list(set(market_obj.string_matrix[i][noncompliance_mask[i]])),
            
            "matched_skills": market_obj.get_matched_skills(i),
            "similarity_match_scores": market_obj.get_matched_skills(i, with_scores=True),

            "not_ideal_skills": list(set(
                    market_obj.string_matrix[i][
                        (market_obj._weighted_match_matrix[i] == 0) &
                        (market_obj.string_matrix[i] != "")
                    ]
                ))
        }
        for i, (job_id, compliance_pct, ideal_compliance_pct) in enumerate(
            zip(market_obj.job_id_by_index, compliance_by_job, ideal_compliance_by_job)
        )
    ]

def market_analysis_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    sorted_analysis = sorted(analysis, key=lambda e: e["minimum_compliance_pct"], reverse=True)
    sorted_counts = [market_obj.skills_count_by_index[market_obj.job_id_by_index.index(e["job_id"])] for e in sorted_analysis]

    df = pd.DataFrame([
        {
            "job_id": entry["job_id"],
            "job_index": entry["job_index"],
            "required_skills": count,
            "minimum_compliance_pct": entry["minimum_compliance_pct"],
            "ideal_compliance_pct": entry["ideal_compliance_pct"],
            "matched_skills": entry["matched_skills"],
            "matched_count": len(entry["matched_skills"]),
            "insufficient_proficiency": entry["not_ideal_skills"],
            "insufficient_count": len(entry["not_ideal_skills"]),
            "nonmatched_skills": entry["nonmatched_skills"],
            "nonmatched_count": entry["nonmatched_skills_count"],
        }
        for entry, count in zip(sorted_analysis, sorted_counts)
    ])
    return df


class MarketSkillsMatrix:
    def __init__(self, skill_type: str, jobs_df: pd.DataFrame):
        if skill_type not in ("hard", "soft"):
            raise ValueError(f"skill_type must be 'hard' or 'soft', got '{skill_type}'")
        if jobs_df.empty:
            raise ValueError("jobs_df cannot be empty")

        self.skill_type = skill_type

        # Populated by _initialize_matrices
        self.string_matrix: np.ndarray = None      # (n_jobs, max_skills) skill descriptions
        self.weight_matrix: np.ndarray = None      # (n_jobs, max_skills) skill weights
        self.embedding_matrix: np.ndarray = None   # (n_jobs, max_skills, vector_dim) embeddings
        self.skills_count_by_index: list[int] = [] # count of skills per job index 
        self.job_id_by_index: list = []            # job_id mapped to matrix row index

        # Populated after combine() / weight_against() calls
        self._match_score_matrix: np.ndarray = None     # raw match counts per (job, skill) cell, starts with 0s and 1s
        self._weighted_match_matrix: np.ndarray = None  # weight-qualified match counts

        self._initialize_matrices(jobs_df)

    def _initialize_matrices(self, jobs_df: pd.DataFrame) -> None:
        matching_jobs_ids = jobs_df["id"].tolist()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = get_position_skills(matching_jobs_ids, table).sort_values("job_id")

        skills_df["index"] = skills_df.groupby("job_id").ngroup()
        max_skills_per_job = skills_df.groupby("index").size().max()

        self.string_matrix = build_padded_matrix(
            skills_df, "skill_description", max_skills_per_job, pad_value=""
        )
        self.weight_matrix = build_padded_matrix(
            skills_df, "weight", max_skills_per_job, pad_value=0, dtype=np.float32
        )
        self.embedding_matrix = build_embedding_matrix(
            skills_df, max_skills_per_job
        )
        self.skills_count_by_index = (
            skills_df["index"].value_counts().sort_index().tolist()
        )
        self.job_id_by_index = (
            skills_df[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
        )
        self._match_score_matrix = create_match_score_matrix(self.skills_count_by_index)
        self._weighted_match_matrix = np.zeros_like(self._match_score_matrix)

    def accumulate_matches(self, match_array: np.ndarray) -> None:
        """Add a match score array into the running match score matrix."""
        if match_array.shape != self._match_score_matrix.shape:
            raise ValueError(
                f"match_array shape {match_array.shape} does not match "
                f"expected {self._match_score_matrix.shape}"
            )
        self._match_score_matrix += match_array

    def accumulate_weighted_matches(
        self, candidate_skill_weight: float, binary_mask: np.ndarray
    ) -> None:
        """Record which job skills are met by a candidate skill at the given weight."""
        if binary_mask.shape != self.weight_matrix.shape:
            raise ValueError(
                f"binary_mask shape {binary_mask.shape} does not match "
                f"weight_matrix shape {self.weight_matrix.shape}"
            )
        candidate_weight_mask = binary_mask * candidate_skill_weight
        weight_qualified = (candidate_weight_mask >= self.weight_matrix) & (self.weight_matrix != 0)
        self._weighted_match_matrix += weight_qualified.astype(np.int8)

    def get_min_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills matched by the candidate, not considering weight.
        """
        qualifying = (self._match_score_matrix > 1).sum(axis=1)  # 0 means padding and n > 1 means a candidate skill matched that job skill n times
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_ideal_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills met at or above their required weight by candidate.
        """
        qualifying = (self._weighted_match_matrix != 0).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_matched_skills(self, job_index: int, top_n: int = 5, with_scores: bool = False):
        scores = self._match_score_matrix[job_index]
        descriptions = self.string_matrix[job_index]
        ranked = sorted(
            ((desc, int(score)) for desc, score in zip(descriptions, scores) if desc != "" and score > 0),
            key=lambda x: x[1],
            reverse=True,
        )
        ranked = ranked[:top_n]
        return ranked if with_scores else [desc for desc, _ in ranked]

### Main Flow

In [ ]:
## Cluster functions
def popular_matches_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    skill_counts: dict[str, int] = {}
    skill_job_counts: dict[str, int] = {}
    skill_embeddings: dict[str, np.ndarray] = {}
    total_jobs = len(analysis)

    for entry in analysis:
        i = entry["job_index"]
        for skill, score in entry["similarity_match_scores"]:
            skill_counts[skill] = skill_counts.get(skill, 0) + score
            skill_job_counts[skill] = skill_job_counts.get(skill, 0) + 1
            if skill not in skill_embeddings:
                mask = market_obj.string_matrix[i] == skill
                if mask.any():
                    skill_embeddings[skill] = market_obj.embedding_matrix[i][mask][0]

    # Cluster semantically similar skills
    skills = list(skill_embeddings.keys())
    embeddings = np.vstack([skill_embeddings[s] for s in skills])
    distances = cosine_distances(embeddings)
    nearest = np.sort(distances, axis=1)[:, 1]
    eps = np.percentile(nearest, 50)
    labels = DBSCAN(eps=eps, min_samples=1, metric='cosine').fit_predict(embeddings)

    # Merge counts by cluster
    cluster_data: dict[int, dict] = {}
    for skill, label in zip(skills, labels):
        if label not in cluster_data:
            cluster_data[label] = {"skills": [], "total_matches": 0, "job_count": 0}
        cluster_data[label]["skills"].append(skill)
        cluster_data[label]["total_matches"] += skill_counts[skill]
        cluster_data[label]["job_count"] += skill_job_counts[skill]

    df = pd.DataFrame([
        {
            "skills": cluster["skills"],
            "total_matches": cluster["total_matches"],
            "job_count": min(cluster["job_count"], total_jobs),  # cap at total_jobs after merging
            "total_jobs": total_jobs,
            "job_coverage_pct": round(100 * min(cluster["job_count"], total_jobs) / total_jobs, 2),
        }
        for cluster in sorted(cluster_data.values(), key=lambda x: x["total_matches"], reverse=True)
    ])

    return df

def nonmatches_display(market_obj: MarketSkillsMatrix) -> pd.DataFrame:
    noncompliance_mask = market_obj._match_score_matrix == 1

    # Extract embeddings and descriptions for nonmatched skills
    missing_skills_matrix = np.where(
        noncompliance_mask[:, :, np.newaxis],
        market_obj.embedding_matrix,
        np.nan
    )
    flat = missing_skills_matrix.reshape(-1, LLM_MODEL_VECTOR_DIMENSIONS)
    valid_mask = ~np.isnan(flat).any(axis=1)
    flat_embeddings = flat[valid_mask]
    flat_descriptions = market_obj.string_matrix[noncompliance_mask]

    # Find eps and cluster
    distances = cosine_distances(flat_embeddings)
    nearest = np.sort(distances, axis=1)[:, 1]
    eps = np.percentile(nearest, 50)
    labels_dbscan = DBSCAN(eps=eps, min_samples=2, metric='cosine').fit_predict(flat_embeddings)

    k = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    labels_kmeans = KMeans(n_clusters=k).fit_predict(flat_embeddings)

    # Group descriptions by cluster
    clusters: dict[str, list[str]] = {}
    outlier_count = 0
    for desc, dbscan_label, kmeans_label in zip(flat_descriptions, labels_dbscan, labels_kmeans):
        if dbscan_label == -1:
            clusters[f"outlier_{outlier_count}"] = [desc]
            outlier_count += 1
        else:
            clusters.setdefault(kmeans_label, []).append(desc)

    return pd.DataFrame([
        {
            "skills": list(set(skills)),
            "total_matches": len(skills),
            "job_coverage_pct": round(100 * len(set(skills)) / len(market_obj.job_id_by_index), 2),
        }
        for skills in sorted(clusters.values(), key=lambda x: len(x), reverse=True)
    ])

In [55]:
RESUME_ID = "ed8a492e-d72b-488d-924a-e198c027aa79"  # REMOVE, must be a parameter
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

jobs_df = filter_job_postings(candidate_industries)

soft_market = MarketSkillsMatrix("soft", jobs_df.copy())
hard_market = MarketSkillsMatrix("hard", jobs_df.copy())

analyze_market(soft_market, candidate_soft_skills_df, "soft")
analyze_market(hard_market, candidate_hard_skills_df, "hard")

market_soft_skills_analysis = get_market_analysis_results(soft_market)
market_hard_skills_analysis = get_market_analysis_results(hard_market)

soft_market_analysis_df = market_analysis_display(soft_market, market_soft_skills_analysis)
hard_market_analysis_df = market_analysis_display(hard_market, market_hard_skills_analysis)

popular_soft_skills_matches = popular_matches_display(soft_market, market_soft_skills_analysis)
popular_hard_skills_matches = popular_matches_display(hard_market, market_hard_skills_analysis)
nonmatched_soft_skills = nonmatches_display(soft_market)
nonmatched_hard_skills = nonmatches_display(hard_market)